
---

## 📚 Sobre este Material

Este material ha sido diseñado con el propósito de **capacitar, actualizar y practicar** conceptos fundamentales de Markdown en Jupyter Notebook. Es una herramienta pensada para facilitar el aprendizaje y la documentación efectiva de proyectos de análisis de datos y ciencia de datos.

### 🤝 Compartir y Colaborar

Este contenido es **libre para compartir, revisar, divulgar y mejorar**. Se promueve activamente su distribución en la comunidad para que más personas puedan beneficiarse y contribuir a su mejora continua. Tu feedback y sugerencias son siempre bienvenidos.

### 👨‍💻 Autor

**Andrés Muñoz**  
*AI & Data Strategy Leader passionate about NLP, LLMs, and MLOps. Driving innovation with data*

- 💼 LinkedIn: [in/amms1989](https://linkedin.com/in/amms1989)
- 🐙 GitHub: [https://github.com/anguihero](https://github.com/anguihero)

---

# Sesión 16: LLMs y NLP Aplicado en Python (100% gratuito)

**Autor:** anmmunozsa@outlook.es · Material de código abierto para compartir y aprender colectivamente.

## 🎯 Objetivo de la sesión
Usar un LLM vía API gratuita (Google Gemini) y un modelo descargado de Hugging Face para resolver tareas reales de NLP: clasificación de texto y reconocimiento de entidades nombradas (NER).

## 🗺️ Tabla de Contenido
1. [Introducción: de CNN a Transformers](#intro)
2. [Fundamentos rápidos de NLP](#fundamentos)
3. [API gratuita de Google Gemini](#gemini)
4. [Prompt Engineering básico](#prompting)
5. [Hugging Face Transformers: clasificación de texto](#hf-clasificacion)
6. [Hugging Face Transformers: NER](#hf-ner)
7. [Comparación: API vs. modelo local](#comparacion)
8. [Ejemplos de aplicación real](#aplicaciones)
9. [Retos de práctica (proyecto integrador)](#retos)


<a id="intro"></a>
## 1. Introducción (para dummies)

En la Sesión 15 entrenamos **nuestra propia** red neuronal desde cero. Los **LLM** (Large Language Models, como Gemini, GPT o Claude) son también redes neuronales — basadas en la arquitectura **Transformer** — pero ya vienen **pre-entrenadas** con enormes cantidades de texto. En vez de entrenarlos nosotros, los **consumimos**: por API (como Gemini) o descargando un modelo más pequeño y especializado (como los de Hugging Face) para correrlo localmente.

<a id="fundamentos"></a>
## 2. Fundamentos Rápidos de NLP

### 🔬 Teoría técnica
- **Tokenización:** dividir el texto en unidades procesables (palabras o sub-palabras).
- **Embeddings:** representar cada token como un vector numérico que captura significado — palabras con significados parecidos quedan "cerca" en ese espacio vectorial.

No profundizaremos en la matemática (queda para quien quiera explorar por su cuenta `background/esp/fundamentos de llm.md`); hoy nos enfocamos en **usar** estos modelos para resolver tareas.

<a id="gemini"></a>
## 3. API Gratuita de Google Gemini

### 🔬 Instructivo paso a paso (sin tarjeta de crédito)

1. Entra a [aistudio.google.com](https://aistudio.google.com) con tu cuenta de Google.
2. Haz clic en **"Get API key"** → **"Create API key"**. Se genera una key gratuita con cuota diaria gratuita.
3. En Colab, guarda la key de forma segura usando **Secrets** (ícono de llave 🔑 en la barra lateral izquierda) con el nombre `GOOGLE_API_KEY`, en vez de escribirla en texto plano en el notebook.
4. Instala el SDK y configura la key.

### 💪 Fortalezas y debilidades
- **Fortaleza:** no requiere GPU propia, respuestas de calidad muy alta, cuota gratuita generosa.
- **Debilidad:** depende de conexión a internet, tiene límites de uso gratuito por minuto/día.

In [ ]:
# !pip install -q google-generativeai

import google.generativeai as genai
from google.colab import userdata  # en Colab, para leer el Secret configurado

GOOGLE_API_KEY = userdata.get("GOOGLE_API_KEY")  # nunca escribas la key directamente en el código
genai.configure(api_key=GOOGLE_API_KEY)

# Nota: el nombre exacto del modelo puede cambiar; revisa los disponibles con genai.list_models()
modelo_gemini = genai.GenerativeModel("gemini-1.5-flash")

respuesta = modelo_gemini.generate_content("Explica en una frase qué es un modelo de lenguaje.")
print(respuesta.text)

### 🧠 Resumen para dummies
`genai.configure(api_key=...)` te "identifica" ante Google. `GenerativeModel(...)` elige el modelo. `generate_content(prompt)` envía tu texto y recibe la respuesta generada.

<a id="prompting"></a>
## 4. Prompt Engineering Básico

### 🔬 Teoría técnica
- **Zero-shot:** pides la tarea directamente, sin ejemplos.
- **Few-shot:** das 1-2 ejemplos de cómo quieres la respuesta, antes de la pregunta real.
- **Chain-of-Thought (CoT):** le pides al modelo "pensar paso a paso", útil en tareas que requieren razonamiento.

In [ ]:
reseñas = [
    "El producto llegó rápido y funciona perfecto, ¡muy feliz!",
    "Pésima calidad, se dañó al segundo día de uso.",
    "Cumple lo que promete, nada extraordinario pero tampoco malo.",
]

# Zero-shot
for reseña in reseñas:
    prompt = f"Clasifica el sentimiento de esta reseña como POSITIVO, NEGATIVO o NEUTRAL. Responde solo con la palabra.\n\nReseña: \"{reseña}\""
    respuesta = modelo_gemini.generate_content(prompt)
    print(f"{reseña[:40]}... -> {respuesta.text.strip()}")

In [ ]:
# Few-shot: le damos ejemplos del formato exacto que queremos
prompt_few_shot = """Clasifica el sentimiento como POSITIVO, NEGATIVO o NEUTRAL.

Reseña: "Excelente atención, todo perfecto" -> POSITIVO
Reseña: "Nunca más vuelvo a comprar aquí" -> NEGATIVO
Reseña: "El producto llegó rápido y funciona perfecto, ¡muy feliz!" ->"""

respuesta = modelo_gemini.generate_content(prompt_few_shot)
print(respuesta.text.strip())

### 🧠 Resumen para dummies
Mientras más claro y estructurado el prompt (y mientras más ejemplos le des si la tarea es ambigua), más consistente será la respuesta del modelo.

<a id="hf-clasificacion"></a>
## 5. Hugging Face Transformers: Clasificación de Texto

### 🔬 Teoría técnica
Hugging Face ofrece modelos pre-entrenados descargables gratis y sin necesidad de API key (para modelos públicos). El objeto `pipeline` simplifica enormemente su uso: elige el modelo, lo descarga, y te da una función lista para usar.

In [ ]:
# !pip install -q transformers

from transformers import pipeline

clasificador = pipeline(
    "sentiment-analysis",
    model="nlptown/bert-base-multilingual-uncased-sentiment",  # soporta español
)

for reseña in reseñas:
    resultado = clasificador(reseña)[0]
    print(f"{reseña[:40]}... -> {resultado['label']} (confianza: {resultado['score']:.2f})")

### 💪 Fortalezas y debilidades (API vs. Hugging Face local)

| | API (Gemini) | Hugging Face local |
|---|---|---|
| Costo | Gratis con límite de cuota | Gratis, sin límite de llamadas |
| Velocidad | Depende de internet | Depende de la GPU/CPU de Colab |
| Flexibilidad | Entiende instrucciones en lenguaje natural | Especializado en la tarea del modelo elegido |
| Control | Menos control sobre el modelo | Control total, se puede re-entrenar/ajustar |

### 🧠 Resumen para dummies
Si necesitas texto generado o razonamiento flexible, usa la API. Si necesitas clasificar o extraer información de forma rápida y repetible (miles de textos), un modelo local de Hugging Face suele ser más práctico.

<a id="hf-ner"></a>
## 6. Hugging Face Transformers: NER (Reconocimiento de Entidades Nombradas)

### 🔬 Teoría técnica
NER identifica y clasifica menciones de personas, organizaciones, lugares, fechas, etc. dentro de un texto. `grouped_entities=True` une automáticamente sub-palabras que forman una sola entidad (ej. "Nueva" + "York" -> "Nueva York").

In [ ]:
extractor_entidades = pipeline(
    "ner",
    model="mrm8488/bert-spanish-cased-finetuned-ner",
    grouped_entities=True,
)

texto_noticia = (
    "Google anunció en Bogotá una inversión de 50 millones de dólares "
    "para su centro de datos en Colombia, según confirmó su gerente Laura Gómez."
)

entidades = extractor_entidades(texto_noticia)
for entidad in entidades:
    print(f"{entidad['word']:20s} -> {entidad['entity_group']} (confianza: {entidad['score']:.2f})")

### 🧠 Resumen para dummies
NER convierte texto libre en datos estructurados: de un párrafo, extraes automáticamente una tabla de "quién, dónde, cuánto".

<a id="comparacion"></a>
## 7. Comparación: API vs. Modelo Local

Repetimos la clasificación de sentimiento de las 3 reseñas con ambos enfoques, lado a lado.

In [ ]:
import pandas as pd

comparacion = []
for reseña in reseñas:
    resultado_hf = clasificador(reseña)[0]
    prompt = f"Clasifica el sentimiento como POSITIVO, NEGATIVO o NEUTRAL. Responde solo con la palabra.\n\nReseña: \"{reseña}\""
    resultado_gemini = modelo_gemini.generate_content(prompt).text.strip()
    comparacion.append({
        "reseña": reseña[:40] + "...",
        "Hugging Face": resultado_hf["label"],
        "Gemini": resultado_gemini,
    })

pd.DataFrame(comparacion)

## 🔎 Laboratorio de profundización: tokens, logits y parámetros de inferencia

Un clasificador produce **logits** y softmax los convierte en probabilidades:

$$p_i=\frac{e^{z_i}}{\sum_j e^{z_j}}$$

Un modelo generativo predice el siguiente token repetidamente. Aquí no entrenamos pesos: configuramos la **inferencia**. `temperature`, `top_p` y máximo de tokens controlan diversidad, muestreo y longitud; no son parámetros aprendidos.


In [ ]:
# Paso 1: inspeccionar tokenización con un modelo pequeño multilingüe
from transformers import AutoTokenizer

tokenizador = AutoTokenizer.from_pretrained(
    "distilbert-base-multilingual-cased"
)
texto_demo = "La analítica avanzada transforma decisiones."
tokens = tokenizador.tokenize(texto_demo)
ids = tokenizador.encode(texto_demo, add_special_tokens=True)
print("Tokens:", tokens)
print("IDs:", ids)


In [ ]:
# Paso 2: truncamiento, padding y tensores del modelo
lote = tokenizador(
    [texto_demo, "Texto corto."],
    padding=True,
    truncation=True,
    max_length=16,
    return_tensors="pt",
)
print(lote.keys())
print("input_ids:", lote["input_ids"].shape)
print("attention_mask:\n", lote["attention_mask"])


### Parámetros y propiedades por tarea

- Tokenizador: `max_length`, `padding`, `truncation`, tokens especiales.
- Pipeline: `device`, `batch_size`, `top_k`, estrategia de agregación NER.
- Generación: `temperature`, `top_p`, `max_output_tokens` y stop sequences, según el SDK vigente.
- Modelo: `model.config` documenta vocabulario, etiquetas y tamaño máximo.

Para clasificación, evalúa datos etiquetados y matriz de confusión; para generación, define una rúbrica. La fluidez no demuestra veracidad. No envíes secretos ni datos personales a una API y valida siempre salidas utilizadas en decisiones.


<a id="aplicaciones"></a>
## 8. Ejemplos de Aplicación en el Mundo Real

- Clasificación automática de tickets de soporte por urgencia/tema.
- Extracción de nombres de personas, empresas y montos de contratos o noticias.
- Chatbots simples y asistentes que responden preguntas frecuentes.
- Resumen automático de documentos largos.

<a id="retos"></a>
## 9. Retos de Práctica (Proyecto Integrador de Cierre)

### 🥉 Reto Básico
Usa la API de Gemini para clasificar el sentimiento de 5 reseñas de producto que tú mismo escribas (no las del ejemplo).

In [ ]:
# Tu solución al Reto Básico aquí


### 🥈 Reto Medio
Usa el pipeline de Hugging Face (`sentiment-analysis`) sobre las mismas 5 reseñas del Reto Básico, y arma una tabla comparando ambos resultados (API vs. modelo local), señalando en qué casos difieren.

In [ ]:
# Tu solución al Reto Medio aquí


### 🥇 Reto Avanzado — Cierre del Curso
1. Aplica `pipeline("ner")` sobre una noticia real (copia un párrafo de cualquier medio) para extraer personas, organizaciones y lugares.
2. Pide a Gemini (con few-shot prompting) que genere un resumen de una sola línea del mismo texto.
3. En una celda de Markdown final, escribe 3-5 líneas reflexionando sobre todo el recorrido del curso: ¿qué tema de las 16 sesiones te pareció más útil para tu contexto de trabajo o estudio, y qué te gustaría explorar por tu cuenta después (`exercises/nlp/`, `exercises/generative_ai/`, `exercises/computer_vision/`)?

In [ ]:
# Tu solución al Reto Avanzado aquí
